# 01 — Feature Engineering

Four signal layers:
1. **Velocity** — rolling tx count + value per card (1h / 24h)
2. **Ratio** — C-column totals, email null flags
3. **Interaction** — new account × high value × express
4. **Network** — email domain mismatch, device/email linkage counts

Output: `data/ieee-fraud-detection/train_engineered.parquet`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../../..") / "data" / "ieee-fraud-detection"
OUT_PATH = DATA_DIR / "train_engineered.parquet"


## 1. Load and join data

In [ ]:
txn = pd.read_csv(DATA_DIR / "train_transaction.csv")
idn = pd.read_csv(DATA_DIR / "train_identity.csv")

df = txn.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["id_01"].notna().astype(np.int8)
print(f"Rows: {len(df):,}  |  identity coverage: {df['has_identity'].mean():.1%}")


## 2. Impute dist1 (59.7% null)

In [ ]:
dist1_median = df["dist1"].median()
df["dist1_missing"] = df["dist1"].isnull().astype(np.int8)
df["dist1"] = df["dist1"].fillna(dist1_median)
print(f"dist1 median used for imputation: {dist1_median:.1f}")
print(f"dist1_missing flag created: {df['dist1_missing'].sum():,} rows flagged")


## 3. Layer 1 — Velocity features

Rolling transaction count and value per card over 1h and 24h windows. Captures burst patterns typical of card testing attacks.

⚠ Slow (~2–3 min) due to per-group iteration.

In [ ]:
def add_velocity_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("TransactionDT").copy()
    for window_secs, label in [(3600, "1h"), (86400, "24h")]:
        counts, values = [], []
        for _, group in df.groupby("card1", sort=False):
            dt = group["TransactionDT"].values
            amt = group["TransactionAmt"].values
            cnt = np.zeros(len(group), dtype=np.int32)
            val = np.zeros(len(group), dtype=np.float32)
            for i in range(len(group)):
                mask = (dt >= dt[i] - window_secs) & (dt < dt[i])
                cnt[i] = mask.sum()
                val[i] = amt[mask].sum()
            counts.append(pd.Series(cnt, index=group.index))
            values.append(pd.Series(val, index=group.index))
        df[f"velocity_count_{label}"] = pd.concat(counts).reindex(df.index)
        df[f"velocity_value_{label}"] = pd.concat(values).reindex(df.index)
    return df

df = add_velocity_features(df)
print("Velocity features added:", [c for c in df.columns if c.startswith("velocity_")])


## 4. Layer 2 — Ratio features

C1–C14 are Vesta-encoded cumulative counts (chargeback history, address/phone usage). Summing them captures overall account history depth.

In [ ]:
def add_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    if c_cols:
        df["c_total"] = df[c_cols].fillna(0).sum(axis=1)
        df["c_max"]   = df[c_cols].fillna(0).max(axis=1)
    df["r_email_missing"] = df["R_emaildomain"].isnull().astype(np.int8)
    df["p_email_missing"] = df["P_emaildomain"].isnull().astype(np.int8)
    return df

df = add_ratio_features(df)
print(f"c_total mean: {df['c_total'].mean():.1f}  |  c_max mean: {df['c_max'].mean():.1f}")


## 5. Layer 3 — Interaction features

D1 = days since first address use, which proxies account age. New accounts making high-value express purchases are a high-risk combination.

In [ ]:
def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    addr_age = df["D1"].fillna(999)   # unknown → treat as established
    new_acct  = addr_age < 7
    high_value = df["TransactionAmt"] > 500
    express    = df["ProductCD"] == "S"

    df["new_acct_highval_express"] = (new_acct & high_value & express).astype(np.int8)
    df["new_acct_flag"]  = new_acct.astype(np.int8)
    df["addr_age_days"]  = addr_age.clip(upper=365)
    return df

df = add_interaction_features(df)
print(f"new_acct_highval_express triggered: {df['new_acct_highval_express'].sum():,} transactions")
print(f"new accounts (<7d): {df['new_acct_flag'].sum():,}")


## 6. Layer 4 — Network / graph features

Devices and email domains shared across many transactions indicate account takeover rings or synthetic identity networks.

In [ ]:
def add_network_features(df: pd.DataFrame) -> pd.DataFrame:
    def email_tld(s: pd.Series) -> pd.Series:
        return s.fillna("").str.split(".").str[-1].str.lower()

    p_tld = email_tld(df.get("P_emaildomain", pd.Series([""] * len(df), index=df.index)))
    r_tld = email_tld(df.get("R_emaildomain", pd.Series([""] * len(df), index=df.index)))
    df["email_domain_mismatch"] = (
        (p_tld != r_tld) & (p_tld != "") & (r_tld != "")
    ).astype(np.int8)

    if "DeviceInfo" in df.columns:
        df["device_linkage_count"] = df["DeviceInfo"].map(
            df["DeviceInfo"].value_counts()
        ).fillna(1).astype(np.int32)
    else:
        df["device_linkage_count"] = 1

    df["email_domain_linkage_count"] = df["P_emaildomain"].map(
        df["P_emaildomain"].value_counts()
    ).fillna(1).astype(np.int32)

    return df

df = add_network_features(df)
print(f"email_domain_mismatch: {df['email_domain_mismatch'].sum():,} transactions")
print(f"device_linkage_count > 100: {(df['device_linkage_count'] > 100).sum():,} transactions")


## 7. Save engineered dataset

In [ ]:
df.to_parquet(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")
print(f"Shape: {df.shape}")

new_features = [
    "velocity_count_1h", "velocity_value_1h", "velocity_count_24h", "velocity_value_24h",
    "c_total", "c_max", "r_email_missing", "p_email_missing",
    "new_acct_highval_express", "new_acct_flag", "addr_age_days",
    "email_domain_mismatch", "device_linkage_count", "email_domain_linkage_count",
    "has_identity", "dist1_missing",
]
print(f"\nEngineered features ({len(new_features)}):")
for f in new_features:
    print(f"  {f}")
